# Seeing Data: completed notebook (solutions)

This executable reference supplies the completed technical work, checked claim audit and suggested written responses. Compare it with your own reasoning after attempting the lab; do not treat it as a substitute for explaining your decisions.

## Suggested learning record

- I expect four datasets with the same summary statistics to be capable of looking very different once plotted.
- A visual pattern is not automatically an insight because interpretation also depends on context, provenance, uncertainty and plausible alternative explanations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 3)


In [ ]:
x_common = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
x_four = [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8]
values = {
    'A': (x_common, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    'B': (x_common, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    'C': (x_common, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    'D': (x_four,   [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
anscombe = pd.concat(
    [pd.DataFrame({'dataset': label, 'x': x, 'y': y}) for label, (x, y) in values.items()],
    ignore_index=True,
)
anscombe.head()


## Shared summaries

**Prediction:** the four summary rows should be almost identical even if their underlying point patterns differ.

In [ ]:
summary_rows = []
for label, group in anscombe.groupby('dataset'):
    slope, intercept = np.polyfit(group['x'], group['y'], 1)
    summary_rows.append({
        'dataset': label,
        'x_mean': group['x'].mean(),
        'y_mean': group['y'].mean(),
        'x_variance': group['x'].var(ddof=1),
        'y_variance': group['y'].var(ddof=1),
        'correlation': group['x'].corr(group['y']),
        'slope': slope,
        'intercept': intercept,
    })
summary = pd.DataFrame(summary_rows)
summary


In [ ]:
assert summary.shape == (4, 8)
assert set(summary['dataset']) == {'A', 'B', 'C', 'D'}
assert np.allclose(summary['x_mean'], 9.0, atol=0.01)
assert np.allclose(summary['y_mean'], 7.5, atol=0.01)
assert np.allclose(summary['correlation'], 0.816, atol=0.01)
print('Summary checks passed.')


## Common-scale small multiples

**Expected output:** four panels with the same axes and nearly identical fitted lines, but visibly different point structures: a broadly linear cloud, a curve, a vertical outlier and a high-leverage horizontal point.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
for ax, (label, group) in zip(axes.flat, anscombe.groupby('dataset')):
    slope, intercept = np.polyfit(group['x'], group['y'], 1)
    line_x = np.array([3, 20])
    ax.scatter(group['x'], group['y'], s=55, color='#18678f')
    ax.plot(line_x, slope * line_x + intercept, color='#e66852', linewidth=2)
    ax.set_title(f'Dataset {label}', fontweight='bold')
    ax.set_xlim(3, 20)
    ax.set_ylim(2, 14)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.suptitle("Anscombe's quartet on common scales", fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()


## Build and check

The `transport` table below is bigger than anything useful to read row by row.

**Prediction:** each mode should show seasonality, a sharp fall during 2020–2021 and a gradual recovery toward its 2019 level by 2025. Trains and buses should remain much larger in absolute rider counts than ferries and light rail.

In [ ]:
# The transport evidence, run this cell, no need to edit it.
# Synthetic monthly rider counts for six NSW regions and four transport modes,
# 2019-2025, with a seasonal cycle and a COVID-shaped shock. Teaching data.
import math, random
random.seed(42)

REGIONS = ["Inner Sydney", "Western Sydney", "Northern Beaches",
           "Central Coast", "Newcastle", "Illawarra"]
MODES = ["Train", "Bus", "Ferry", "Light rail"]
BASE = {"Train": 1_000_000, "Bus": 700_000, "Ferry": 90_000, "Light rail": 120_000}
FACTOR = {"Inner Sydney": 1.3, "Western Sydney": 1.1, "Northern Beaches": 0.55,
          "Central Coast": 0.45, "Newcastle": 0.5, "Illawarra": 0.42}

rows = []
for region in REGIONS:
    for mode in MODES:
        base = BASE[mode] * FACTOR[region]
        for date in pd.date_range("2019-01-01", "2025-12-01", freq="MS"):
            season = 1 + 0.08 * math.sin((date.month - 1) / 12 * 2 * math.pi)
            covid = 1.0
            if pd.Timestamp("2020-03-01") <= date <= pd.Timestamp("2021-12-01"):
                covid = 0.35 + 0.3 * (date - pd.Timestamp("2020-03-01")).days / 640
            elif date > pd.Timestamp("2021-12-01"):
                covid = min(1.0, 0.65 + 0.35 * (date - pd.Timestamp("2021-12-01")).days / 1100)
            rows.append({"date": date, "region": region, "mode": mode,
                         "riders": int(base * season * covid * random.gauss(1, 0.03))})

transport = pd.DataFrame(rows)
print(f"{len(transport):,} rows")
transport.head()

In [ ]:
def plot_mode_recovery(transport: pd.DataFrame) -> pd.DataFrame:
    agg = transport.groupby(["date", "mode"], as_index=False)["riders"].sum()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for mode, g in agg.groupby("mode"):
        ax.plot(g["date"], g["riders"] / 1e6, label=mode)
    ax.set_ylabel("riders (millions)")
    ax.set_title("Patronage collapsed in 2020-21 and has largely recovered")
    ax.legend()
    plt.show()
    return agg


plotted = plot_mode_recovery(transport)
plotted.head()

In [ ]:
# Checks: totals reconcile, nothing missing, one hand-derivable spot value.
# TRACE: the plotted table must reconcile with the source table.
assert plotted["riders"].sum() == transport["riders"].sum(), (
    "Aggregation lost or duplicated riders: plotted total != source total")
# CHECK: every mode present, every month present, no NaNs.
assert set(plotted["mode"]) == set(MODES), "A mode went missing in the aggregation"
assert plotted.groupby("mode")["date"].nunique().eq(84).all(), (
    "Each mode should have 84 monthly points (2019-01..2025-12)")
assert plotted["riders"].notna().all(), "NaNs appeared during aggregation"
# Spot total, hand-derivable: Train riders in Jan 2019 across all six regions.
jan = pd.Timestamp("2019-01-01")
jan_train = plotted[(plotted["mode"] == "Train") & (plotted["date"] == jan)]["riders"].iloc[0]
source_jan_train = transport[(transport["mode"] == "Train") & (transport["date"] == jan)]["riders"].sum()
assert jan_train == source_jan_train, "Spot total disagrees for Train, Jan 2019"
print("TRACE ✓  CHECK ✓, now TEST: does the recovery story survive a per-region view?")

In [ ]:
# Evidence used in the claim audit below. Values are percentages of each mode's 2019 annual total.
annual_by_mode = (
    plotted.assign(year=plotted["date"].dt.year)
    .groupby(["year", "mode"], as_index=False)["riders"].sum()
)
annual_pivot = annual_by_mode.pivot(index="year", columns="mode", values="riders")
annual_vs_2019 = annual_pivot.div(annual_pivot.loc[2019]).mul(100).round(1)
absolute_drop_2019_2020 = (annual_pivot.loc[2019] - annual_pivot.loc[2020]).sort_values(ascending=False)
proportional_drop_2019_2020 = (100 - annual_vs_2019.loc[2020]).sort_values(ascending=False)

network_monthly = plotted.groupby("date", as_index=False)["riders"].sum()
latest_network = network_monthly.iloc[-1]
peak_network = network_monthly.loc[network_monthly["riders"].idxmax()]

assert annual_vs_2019.loc[2020].between(50, 52).all()
assert absolute_drop_2019_2020.index[0] == "Train"
assert proportional_drop_2019_2020.index[0] == "Light rail"
assert annual_vs_2019.loc[2023, "Ferry"] < 100
assert annual_vs_2019.loc[2025, "Bus"] > 99
assert peak_network["date"] < latest_network["date"]
assert peak_network["riders"] > latest_network["riders"]

print(annual_vs_2019)
print("\nAbsolute rider loss, 2019 to 2020:\n", absolute_drop_2019_2020)
print("\nPercentage loss, 2019 to 2020:\n", proportional_drop_2019_2020)
print(f"Latest network month: {latest_network['date']:%Y-%m}, {latest_network['riders']:,} riders")
print(f"Peak network month: {peak_network['date']:%Y-%m}, {peak_network['riders']:,} riders")

## Anscombe interpretation

- A resembles a conventional linear relationship.
- B is curved, so the fitted line misses systematic structure.
- C is strongly influenced by one vertical outlier.
- D is strongly influenced by one high-leverage horizontal point.
- Similar summary statistics do not establish similar data-generating structure.

Suggested repair of the interpretation prose:

> The datasets share similar summary statistics and fitted lines, but their plots reveal different structures, including curvature and influential outliers; the summaries alone do not justify a common model.

## Transport claim audit

The original prose contains compound and ambiguous wording. The audit below splits it into precise claims; in particular, ‘hardest hit’ is tested as both an absolute and a proportional decline.

| Claim | Classification | Evidence or reason |
|---|---|---|
| Ridership collapsed across all modes in 2020. | **Supported directly** | `annual_vs_2019.loc[2020]` shows every mode at about 50–52% of its 2019 total. |
| Trains had the largest absolute rider loss from 2019 to 2020. | **Supported directly** | `absolute_drop_2019_2020` ranks trains first because their baseline volume is largest. |
| Trains had the largest proportional loss from 2019 to 2020. | **Contradicted** | `proportional_drop_2019_2020` ranks light rail first (49.6% lost) and trains third (48.7% lost). This shows why ‘hardest hit’ must be defined. |
| Ferries recovered fastest. | **Plausible but unverified as written** | ‘Recovered’ has no stated threshold. Annual totals, return above the 2019 monthly average and comparison with the same calendar month can produce different answers. |
| Ferries exceeded their 2019 annual total by 2023. | **Contradicted** | Ferry patronage in `annual_vs_2019` is 82.7% of 2019 in 2023. |
| Returning office commuters drove the recovery. | **Plausible but unverified** | The table has no journey-purpose or office-attendance field, so it cannot test the proposed cause. |
| Bus ridership stabilised at about 90% of 2019. | **Contradicted** | The annual comparison rises from 94.1% in 2024 to 100.3% in 2025 rather than stabilising around 90%. |
| The latest month carries more passengers than any earlier point. | **Contradicted** | `network_monthly` identifies April 2019 as the peak; December 2025 is lower. |
| Weekend travel now outweighs weekday commuting. | **Unsupported** | The data are monthly totals and contain neither day type nor journey purpose. |

The supported claims name the exact outputs that back them. The remaining classifications either use those outputs to show contrary evidence or identify definitions and variables that the dataset does not supply.

## Deliberately misleading view and repair

Both panels below use exactly the same Dataset C rows. Only the visible y-range changes.

In [ ]:
dataset_c = anscombe.query("dataset == 'C'")

misleading_fig, misleading_ax = plt.subplots(figsize=(5.5, 4.2))
misleading_ax.scatter(dataset_c['x'], dataset_c['y'], color='#e66852', s=55)
misleading_ax.set(xlim=(3, 20), ylim=(4, 10), title='Misleading crop: outlier omitted', xlabel='x', ylabel='y')
misleading_fig.tight_layout()
plt.show()

repaired_fig, repaired_ax = plt.subplots(figsize=(5.5, 4.2))
repaired_ax.scatter(dataset_c['x'], dataset_c['y'], color='#18678f', s=55)
repaired_ax.set(xlim=(3, 20), ylim=(2, 14), title='Repair: full common scale', xlabel='x', ylabel='y')
repaired_fig.tight_layout()
plt.show()


## Repair explanation

- **What changed visually:** the misleading panel crops the y-axis to 4–10, hiding Dataset C's point at y = 12.74; the repaired panel restores the common 2–14 scale.
- **False impression made easier:** without the influential high point, the remaining marks can appear like an unremarkable compact cloud, concealing why the fitted summary is fragile.
- **Why the repair is fairer:** restoring the full range makes every observation visible and uses the same scale as the four-panel comparison, so the audience can assess the outlier rather than having it silently removed.

## Suggested exit ticket

- The transport figure invites the audience to believe that all four modes experienced a large 2020–2021 decline followed by recovery toward their 2019 levels.
- The most important limitation is that these are synthetic monthly teaching data aggregated across six regions; they contain no passenger characteristics, journey purpose, service level or causal variables.
- Before sharing it, I would verify the data-generating assumptions, aggregation across all regions, completeness of every monthly series, and comparisons of 2025 with the 2019 baseline.

## AI disclosure for this published solution

| Question | Answer |
|---|---|
| Which AI assistant(s) did you use, if any? | OpenAI Codex assisted with the final review of this instructor solution. |
| What did the assistant contribute? | It checked the code, calculated the claim-audit evidence, identified omitted responses and helped draft the completed explanatory text. |
| What did you accept, modify or reject, and why? | The final text was matched to the actual deterministic outputs; broad or causal statements not supported by the fields were classified or qualified accordingly. |
| How did you verify the output? | Every code cell was executed top-to-bottom, all assertions passed, the annual comparisons were independently recomputed, and the notebook structure was checked against the student lab. |
| What limitations or unverified claims remain? | The transport data are synthetic and monthly, so the notebook cannot establish real-world causes, passenger purposes, or weekday-versus-weekend behaviour. |